In [1]:
# Install deps (run once):
!pip install google-api-python-client youtube-transcript-api isodate pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 9.8 MB/s  0:00:01m 10.2 MB/s eta 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.5
    Uninstalling protobuf-4.25.5:
      Successfully uninstalled protobuf-4.25.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [google-api-python-client]32m8/9 [google-api-python-client]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
tf-nightly 2.19.0.dev20241121 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
wandb 0.18.7 requires protobuf!=4.21.0,!=5.2

In [1]:
# Notebook-ready: Ransomware transcript capture pipeline
# ------------------------------------------------------

import re
import csv
import json
from pathlib import Path
from typing import List, Dict, Optional, Iterable, Tuple

import pandas as pd
import isodate
from googleapiclient.discovery import build
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
    CouldNotRetrieveTranscript,
)


/home/henrykabuye/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:

from googleapiclient.errors import HttpError

def _list_video_ids_by_search(youtube, channel_uc: str, max_videos: int) -> List[str]:
    """Fallback: list newest videos via search endpoint."""
    video_ids = []
    page_token = None
    while len(video_ids) < max_videos:
        resp = youtube.search().list(
            part="id",
            channelId=channel_uc,
            order="date",
            type="video",
            maxResults=min(50, max_videos - len(video_ids)),
            pageToken=page_token
        ).execute()

        for item in resp.get("items", []):
            vid = item.get("id", {}).get("videoId")
            if vid:
                video_ids.append(vid)

        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return video_ids
def _sanitize(s: str, max_len: int = 80) -> str:
    s = re.sub(r"[^\w\- ]+", "", s, flags=re.UNICODE).strip()
    s = re.sub(r"\s+", "_", s)
    return s[:max_len] if len(s) > max_len else s

def _parse_channel_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    """Returns (UC_channel_id, handle) if present in URL."""
    if not isinstance(url, str):
        return None, None
    url = url.strip()
    m = re.search(r"(UC[a-zA-Z0-9_-]{20,})", url)
    uc = m.group(1) if m else None
    m = re.search(r"/@([a-zA-Z0-9_.-]+)", url)
    handle = m.group(1) if m else None
    return uc, handle

def _chunked(seq: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def _resolve_channel_uc(youtube, channel_url: str, channel_name: str) -> Optional[str]:
    """
    Best-effort channel resolution:
      1) UC... in URL
      2) @handle via channels.list(forHandle=...)
      3) search by channel_name (ambiguous fallback)
    """
    uc, handle = _parse_channel_url(channel_url)
    if uc:
        return uc

    if handle:
        try:
            resp = youtube.channels().list(part="id", forHandle=handle).execute()
            items = resp.get("items", [])
            if items:
                return items[0]["id"]
        except Exception:
            pass

    # Fallback: channel search (can be ambiguous)
    try:
        resp = youtube.search().list(
            part="snippet",
            q=channel_name,
            type="channel",
            maxResults=1,
        ).execute()
        items = resp.get("items", [])
        if items:
            return items[0]["snippet"]["channelId"]
    except Exception:
        pass

    return None

def _get_uploads_playlist_id(youtube, channel_uc: str) -> Optional[str]:
    resp = youtube.channels().list(part="contentDetails", id=channel_uc).execute()
    items = resp.get("items", [])
    if not items:
        return None
    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

def _list_video_ids_from_uploads(youtube, uploads_playlist_id: str, max_videos: int) -> List[str]:
    """Newest-first video IDs from a channel uploads playlist."""
    video_ids: List[str] = []
    page_token = None

    while len(video_ids) < max_videos:
        resp = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=min(50, max_videos - len(video_ids)),
            pageToken=page_token,
        ).execute()

        for item in resp.get("items", []):
            video_ids.append(item["contentDetails"]["videoId"])

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    return video_ids

def _get_video_metadata(youtube, video_ids: List[str]) -> List[Dict]:
    """Fetch snippet + contentDetails (duration) for a batch of videos."""
    rows: List[Dict] = []
    for batch in _chunked(video_ids, 50):
        resp = youtube.videos().list(part="snippet,contentDetails", id=",".join(batch)).execute()
        for v in resp.get("items", []):
            snippet = v["snippet"]
            cd = v["contentDetails"]
            title = snippet.get("title", "")
            description = snippet.get("description", "")
            published_at = snippet.get("publishedAt", "")
            year = int(published_at[:4]) if published_at else None
            dur_iso = cd.get("duration", "PT0S")
            dur_seconds = int(isodate.parse_duration(dur_iso).total_seconds())
            rows.append({
                "video_id": v["id"],
                "title": title,
                "description": description,
                "published_at": published_at,
                "year": year,
                "duration_iso": dur_iso,
                "duration_seconds": dur_seconds,
            })
    return rows

def _keyword_match(text: str, keywords: List[str]) -> bool:
    if not keywords:
        return True
    t = (text or "").lower()
    return any(k in t for k in keywords)

def _fetch_transcript(video_id: str, prefer_langs: List[str]) -> Optional[str]:
    """Return transcript text if available, else None."""
    try:
        for lang in prefer_langs:
            try:
                transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
                return " ".join([t["text"] for t in transcript])
            except Exception:
                pass

        transcript = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([t["text"] for t in transcript])

    except (TranscriptsDisabled, NoTranscriptFound, CouldNotRetrieveTranscript):
        return None
    except Exception:
        return None

def run_ransomware_transcript_capture(
    api_key: str,
    registry_xlsx: str,
    out_dir: str = "out_transcripts",
    sheet: Optional[str] = None,
    included_only: bool = True,
    min_minutes: float = 5.0,
    per_channel: int = 10,
    scan_recent: int = 80,
    keywords: Optional[List[str]] = None,
    match_description: bool = True,
    prefer_langs: Optional[List[str]] = None,
    start_index: int = 1,
):
    """
    Reads a channel registry and saves transcripts for up to N ransomware-relevant videos per channel.
    
    Registry must contain columns:
      - Channel_ID
      - Channel_Name
      - Channel_URL
    Optional:
      - Included (Yes/No)
    
    Returns:
      manifest_df (pandas DataFrame)
    """
    if keywords is None:
        keywords = [
            "ransomware", "extortion", "double extortion", "locker",
            "encrypt", "encryption", "decrypt", "data leak",
            "incident response", "ir",
            "lockbit", "alphv", "blackcat", "conti", "ryuk",
            "cl0p", "clop", "akira", "revil", "sodinokibi",
        ]
    keywords = [k.strip().lower() for k in keywords if k.strip()]

    if prefer_langs is None:
        prefer_langs = ["en", "en-GB", "en-US"]

    min_seconds = int(min_minutes * 60)

    youtube = build("youtube", "v3", developerKey=api_key)
    df = pd.read_excel(registry_xlsx, sheet_name=sheet)

    required_cols = {"Channel_ID", "Channel_Name", "Channel_URL"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Registry missing required columns: {sorted(missing)}")

    if included_only and "Included (Yes/No)" in df.columns:
        df = df[df["Included (Yes/No)"].astype(str).str.strip().str.lower().eq("yes")].copy()

   
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    
    manifest_rows = []
    
    # --- AUTO-DETECT NEXT AVAILABLE V NUMBER ---
    existing = [p.stem for p in out_path.glob("V*.txt")]
    nums = [int(s[1:]) for s in existing if s[1:].isdigit()]
    global_counter = (max(nums) + 1) if nums else start_index
    # --------------------------------------------


    for _, row in df.iterrows():
        channel_id = str(row["Channel_ID"]).strip()
        channel_name = str(row["Channel_Name"]).strip()
        channel_url = str(row["Channel_URL"]).strip()

        channel_uc = _resolve_channel_uc(youtube, channel_url, channel_name)
        if not channel_uc:
            print(f"[WARN] Could not resolve channel UC id for {channel_id} | {channel_name}")
            continue

        uploads = _get_uploads_playlist_id(youtube, channel_uc)
        if not uploads:
            print(f"[WARN] Could not get uploads playlist for {channel_id} ({channel_uc})")
            continue

        scan_n = max(scan_recent, per_channel)
        video_ids = _list_video_ids_from_uploads(youtube, uploads, max_videos=scan_n)
        if not video_ids:
            print(f"[WARN] No videos found for {channel_id} ({channel_uc})")
            continue

        meta = _get_video_metadata(youtube, video_ids)

        # Filter duration >= min_seconds
        meta = [m for m in meta if m["duration_seconds"] >= min_seconds]

        # Keyword filter
        selected = []
        for m in meta:
            title = m.get("title", "")
            desc = m.get("description", "")
            text = title + ("\n" + desc if match_description else "")
            if _keyword_match(text, keywords):
                t = text.lower()
                matched = sorted({k for k in keywords if k in t})
                m["matched_keywords"] = ";".join(matched)
                selected.append(m)

        # keep newest-first
        selected = selected[:per_channel]

        channel_folder = out_path / f"{channel_id}_{_sanitize(channel_name)}"
        channel_folder.mkdir(parents=True, exist_ok=True)

        kept = 0
        for m in selected:
            yt_vid = m["video_id"]
            title = m["title"]
            year = m["year"]
            published_at = m["published_at"]
            dur_s = m["duration_seconds"]
            dur_iso = m["duration_iso"]
            matched_kw = m.get("matched_keywords", "")

            transcript_text = _fetch_transcript(yt_vid, prefer_langs)
            transcript_available = transcript_text is not None

            vid_id = f"V{global_counter:04d}"
            

            txt_file = channel_folder / f"{vid_id}.txt"
            meta_file = channel_folder / f"{vid_id}.meta.json"

            # txt_file = out_path / f"{vid_id}.txt"
            # meta_file = out_path / f"{vid_id}.meta.json"


            if transcript_text is None:
                transcript_text = "[NO TRANSCRIPT AVAILABLE]\n"
            if txt_file.exists():
                print(f"[SKIP] {vid_id} exists: {txt_file}")
                continue

            txt_file.write_text(transcript_text, encoding="utf-8")

            meta_payload = {
                "Video_ID": vid_id,
                "Channel_ID": channel_id,
                "Channel_Name": channel_name,
                "Channel_UC": channel_uc,
                "YouTube_Video_ID": yt_vid,
                "Video_Title": title,
                "Year": year,
                "PublishedAt": published_at,
                "DurationSeconds": dur_s,
                "DurationISO": dur_iso,
                "Matched_Keywords": matched_kw,
                "Transcript_Available": transcript_available,
            }
            meta_file.write_text(json.dumps(meta_payload, indent=2), encoding="utf-8")
            global_counter += 1
            manifest_rows.append({
                **meta_payload,
                "Transcript_Path": str(txt_file),
                "Meta_Path": str(meta_file),
            })

            kept += 1

        print(f"[OK] {channel_id} | saved {kept} transcript(s) -> {out_path}")


    manifest_df = pd.DataFrame(manifest_rows)
    manifest_csv = out_path / "manifest.csv"
    manifest_df.to_csv(manifest_csv, index=False, encoding="utf-8")

    print(f"\nDone. Manifest saved to: {manifest_csv}")
    return manifest_df


In [ ]:
YOUR_YOUTUBE_API_KEY

In [3]:
manifest = run_ransomware_transcript_capture(
    api_key="My_key",
    registry_xlsx="rubrics/Dataset_Channel_Registry_Updated_50_fixed_urls.xlsx",
    sheet="Channel_Registry",
    out_dir="transcripts",
    start_index=1,
    included_only=True,
    min_minutes=5,
    per_channel=10,
    scan_recent=80,
    match_description=True
)

manifest.head()

[OK] C01 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C02 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C03 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C04 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C05 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C06 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C07 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C08 | saved 10 transcript(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts
[OK] C09 | saved 10 tran

,Video_ID,Channel_ID,Channel_Name,Channel_UC,YouTube_Video_ID,Video_Title,Year,PublishedAt,DurationSeconds,DurationISO,Matched_Keywords,Transcript_Available,Transcript_Path,Meta_Path
0,V0970,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,OfwHWjZvvuE,Blurring the Lines: Intrusion Shows Connection...,2025,2025-09-08T14:38:04Z,382,PT6M22S,ir;ransomware,False,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
1,V0971,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,5fu-wYbBvuY,Hide Your RDP: Password Spray Leads to RansomH...,2025,2025-06-30T00:52:31Z,343,PT5M43S,ir;ransomware,False,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
2,V0972,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,7HD-RYqIm3Q,Another Confluence Bites the Dust: Falling to ...,2025,2025-05-19T00:46:35Z,365,PT6M5S,ir;ransomware,False,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
3,V0973,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,hpgjomPwHp4,Fake Zoom Ends in BlackSuit Ransomware,2025,2025-03-31T00:30:19Z,371,PT6M11S,ir;ransomware,False,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
4,V0974,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,BMTtJLEJRGQ,Confluence Exploit Leads to LockBit Ransomware,2025,2025-02-24T00:33:55Z,341,PT5M41S,ir;lockbit;ransomware,False,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
